# Synthetic sensor-fault benchmark generator

**Notebook scope:** reproducibly generate and verify sensor-level synthetic faults using the configured dataset splits and the supplied 30-step pseudocode. **Model training, thresholds, performance metrics, and model comparisons are intentionally deferred.**

**Data roles:** `TRAIN_PATH`, `VALIDATION_PATH`, and `TEST_PATH` load distinct source files. The benchmark generator currently uses **training data** (`BENCHMARK_SPLIT = "train"`) as its source. Normal rows from the training data are used to estimate reference sensor standard deviations, regardless of the benchmark source selected.

### What this notebook produces

- `manifest.csv`: one experiment per row, including the source split, sensor, fault type, severity, duration, timestamps, replicate, and deterministic random seed.
- `examples/*.csv`: a few complete example benchmark datasets by default; optionally one dataset per experiment in `benchmarks/*.csv`.
- `verification_summary.csv`: checks for unintended changes to the benchmark source.
- `reading_window_alignment_preview.csv`: a small label-alignment example for future anomaly detection (not model predictions).

All three input CSV files remain unchanged. Each experiment changes a *fresh copy* of the selected source in **one of the first three sensor columns**. The fourth feature, `Oxygen over threshold`, remains unchanged (a binary feature), as does the final `PW_0.5h` future-failure target (0 = no failure within the next 30 minutes; 1 = failure within the next 30 minutes).

**Important:** A positive `PW_0.5h` label describes an *upcoming* machine failure, not necessarily a malfunction at the current timestamp. It is different from an injected sensor-level anomaly (`synthetic_anomaly = 1`). For this conservative pilot benchmark, injected faults are restricted to intervals labeled `PW_0.5h = 0`, with a preceding label-zero reading-window history. Label 0 is only a baseline candidate, not proof of fault-free operation. Never use `PW_0.5h` as an Isolation Forest input feature; doing so would leak future outcomes. Because the default benchmark source is **train**, these generated examples are suitable for checking the fault generator, but **not for an unbiased final evaluation of a model fitted on the same training data**. Switch `BENCHMARK_SPLIT` to `"test"` for final held-out benchmarking.

## 1. Setup and configurable paths

Dependencies: `pandas`, `numpy`, `matplotlib` and a Jupyter Python kernel. If needed, install with `pip install pandas numpy matplotlib jupyter nbformat` in your environment.
**No model is fitted in this notebook.**

In [ ]:
from pathlib import Path
from itertools import product
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ---- Edit this cell to control the run ----
DATA_FOLDER = "PW_0.5h"
OUTPUT_DIR = Path("synthetic_benchmark_outputs")
TRAIN_PATH = f"{DATA_FOLDER}/train.csv"
VALIDATION_PATH = f"{DATA_FOLDER}/val.csv"
TEST_PATH = f"{DATA_FOLDER}/test.csv"
BENCHMARK_SPLIT = "train"       # "train", "validation", or "test"; train by request

MODE = "pilot"                 # "pilot"or "full"
RANDOM_SEED = 42
PILOT_REPLICATES = 2
FULL_REPLICATES = 20           # increase to 30 if desired
FULL_SEVERITIES_SD = (0.5, 1.0, 2.0, 3.0, 5.0)
FULL_DURATIONS_SAMPLES = (1, 3, 5, 10, 30, 60)
BENIGN_NOISE_SEVERITIES_SD = (0.02, 0.05, 0.10)
PILOT_SEVERITY_SD = 2.0
PILOT_DURATION_SAMPLES = 5
READING_WINDOWS = (5, 10, 30, 60)  # in readings = minutes for this dataset

SAVE_ALL_BENCHMARK_CSV = False  # WARNING: "full" can create many GB of files
SAVE_EXAMPLE_CSV = True
PLOT_EXAMPLES = True

SENSOR_COLS = (
    "CMS air pressure",
    "Concentrazione ossigeno base",
    "Nitrogen pressure",
)
TIMESTAMP_COL = "timestamp"
FAILURE_HORIZON_COL = "PW_0.5h"  # 1 = failure within the next 30 minutes; 0 = no such failure
BINARY_FEATURE_COL = "Oxygen over threshold"
FAULT_TYPES = (
    "gaussian_noise", "spike", "short_burst", "random_corruption", "dropout",
    "bias", "drift", "stuck_sensor", "increased_noise", "intermittent_fault",
)

assert MODE in ("pilot", "full")
assert BENCHMARK_SPLIT in ("train", "validation", "test")
assert RANDOM_SEED >= 0 and PILOT_REPLICATES > 0 and FULL_REPLICATES > 0
assert all(w >= 1 for w in READING_WINDOWS)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load and validate all three splits

Sensor readings are used **as supplied**; no assumptions are made about their physical units. All three files must exist under `DATA_FOLDER` and have the configured column names, with `PW_0.5h` (a future-failure label) as their last column. A value of 1 signals a failure within the **next 0.5 hours**, not a presently observed abnormal machine state.

The table below summarizes **training data**. The selected synthetic benchmark source is controlled separately through `BENCHMARK_SPLIT` (default: `train`); validation and test data are retained for later ML evaluation.

In [ ]:
def load_and_validate(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Missing input file: {path.resolve()}")
    df = pd.read_csv(path)
    expected = [TIMESTAMP_COL, *SENSOR_COLS, BINARY_FEATURE_COL]
    missing = [col for col in expected if col not in df.columns]
    if missing:
        raise ValueError(f"{path.name}: missing required columns {missing}")
    if df.columns[-1] != FAILURE_HORIZON_COL:
        raise ValueError(f"The final column must be {FAILURE_HORIZON_COL!r}.")
    if FAILURE_HORIZON_COL not in df:
        raise ValueError("The dataset needs a binary 0/1 future-failure label.")
    df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL], errors="raise")
    if df[TIMESTAMP_COL].isna().any() or not df[TIMESTAMP_COL].is_monotonic_increasing:
        raise ValueError(f"{path.name}: timestamps must be valid and increasing.")
    if df[TIMESTAMP_COL].duplicated().any():
        raise ValueError(f"{path.name}: duplicate timestamps detected.")
    for col in SENSOR_COLS:
        df[col] = pd.to_numeric(df[col], errors="raise")
        if not np.isfinite(df[col].to_numpy(dtype=float)).all():
            raise ValueError(f"{path.name}: sensor {col} has missing or infinite values.")
    if FAILURE_HORIZON_COL in df:
        if not df[FAILURE_HORIZON_COL].isin([0, 1]).all():
            raise ValueError(f"{path.name}: future-failure label must be binary 0/1.")
        df[FAILURE_HORIZON_COL] = df[FAILURE_HORIZON_COL].astype('int8')
    return df.reset_index(drop=True)

clean_test = load_and_validate(TEST_PATH)
clean_train = load_and_validate(TRAIN_PATH)
clean_validation = load_and_validate(VALIDATION_PATH)

source_splits = {"train": clean_train, "validation": clean_validation, "test": clean_test}
clean_benchmark = source_splits[BENCHMARK_SPLIT]
print(f"Synthetic benchmark source: {BENCHMARK_SPLIT} ({len(clean_benchmark):,} readings)")

sensor_summary = pd.DataFrame({
    "min": clean_train[list(SENSOR_COLS)].min(),
    "max": clean_train[list(SENSOR_COLS)].max(),
    "std_all": clean_train[list(SENSOR_COLS)].std(),
    "distinct_values": clean_train[list(SENSOR_COLS)].nunique(),
})
print(f"Rows: {len(clean_train):,}; no failure within 30 min (0): {(clean_train[FAILURE_HORIZON_COL]==0).sum():,}; "
      f"failure within 30 min (1): {(clean_train[FAILURE_HORIZON_COL]==1).sum():,}")
print(f"Time range: {clean_train[TIMESTAMP_COL].min()} through {clean_train[TIMESTAMP_COL].max()}")
print("Missing-value counts:", clean_train.isna().sum().to_dict())
display(sensor_summary)
display(clean_train.head(4))


## 3. Baseline SD and eligible injection intervals

Fault intensity is expressed in multiples of each sensor's **standard deviation (SD)**, estimated from **training rows labeled `PW_0.5h = 0`**. Those rows are *baseline candidates* (no failure forecast within the next 30 minutes), **not confirmed fault-free samples**; gradual deterioration can begin earlier than the label horizon.

Faults are placed in **`clean_benchmark`**, which defaults to `clean_train`. As a conservative design choice for evaluating *synthetic sensor-level* faults independently of impending machine failures, the injection interval and preceding `max(READING_WINDOWS)-1` readings must have `PW_0.5h = 0` and contiguous one-minute timestamps. This filter excludes labeled pre-failure periods; it does not establish current machine health. The future-failure column is metadata/ground truth, **not a model input feature**.


In [ ]:
reference_source = "train (PW_0.5h=0 baseline candidates)"
reference_df = clean_train.loc[clean_train[FAILURE_HORIZON_COL].eq(0)]  # excludes labeled imminent-failure periods
if len(reference_df) < 3:
    raise ValueError("Insufficient PW_0.5h=0 training readings to estimate sensor standard deviations.")
REFERENCE_STATS = {
    col: {
        "mean": float(reference_df[col].mean()),
        "std": float(reference_df[col].std(ddof=1)),
        "min": float(reference_df[col].min()),
        "max": float(reference_df[col].max()),
    }
    for col in SENSOR_COLS
}
if any(not np.isfinite(s["std"]) or s["std"] <= 0 for s in REFERENCE_STATS.values()):
    raise ValueError("Some PW_0.5h=0 baseline sensors have zero/invalid SD. Adjust baseline data.")
print("Reference source:", reference_source)
display(pd.DataFrame(REFERENCE_STATS).T)

outside_failure_horizon = clean_benchmark[FAILURE_HORIZON_COL].to_numpy() == 0
steps = clean_benchmark[TIMESTAMP_COL].diff().dt.total_seconds().to_numpy()
regular_step = np.r_[True, np.isclose(steps[1:], 60.0)]
# Exclude readings with positive pre-failure labels and positions after timestamp gaps.
bad = ~outside_failure_horizon | ~regular_step
bad_prefix = np.r_[0, np.cumsum(bad.astype(np.int64))]
lookback = max(READING_WINDOWS) - 1

def eligible_starts(duration):
    """Starts with PW_0.5h=0 throughout contiguous lookback and injection interval."""
    duration = int(duration)
    if duration <= 0 or duration > len(clean_benchmark):
        return np.array([], dtype=np.int64)
    starts = np.arange(lookback, len(clean_benchmark) - duration + 1, dtype=np.int64)
    valid = bad_prefix[starts + duration] == bad_prefix[starts - lookback]
    return starts[valid]

eligible_cache = {d: eligible_starts(d) for d in FULL_DURATIONS_SAMPLES if d <= len(clean_benchmark)}
print("Eligible starts (no positive failure-horizon labels in lookback):", {k:len(v) for k,v in eligible_cache.items()})

## 4. Fault definitions and labeling rules

| Fault | Injection mechanism | `synthetic_anomaly` label |
|---|---|---|
| `gaussian_noise` | Low-amplitude Gaussian jitter, SD × 0.02–0.10 | **0** — benign *negative control* (not a labeled fault) |
| `spike` | ± severity × SD on a single reading | One reading = 1 |
| `short_burst` | Consecutive spikes with constant signed offset | Every active reading = 1 |
| `random_corruption` | Replace readings by randomly drawn outlying values around reference mean | Every active reading = 1 |
| `dropout` | Replace readings by `NaN` | Every active reading = 1 |
| `bias` | Persistent ± severity × SD offset | Every active reading = 1 |
| `drift` | Gradually increasing signed offset | Every active reading = 1 |
| `stuck_sensor` | Hold previous reading's value (may coincide with some unmodified values) | Entire stuck interval = 1 |
| `increased_noise` | Add higher-variance Gaussian noise | Every active reading = 1 |
| `intermittent_fault` | Alternate between faulty and untouched readings | **Only actually faulty phases** = 1 |

The extra column `injection_interval` marks the full scheduled interval even when a sensor temporarily appears normal. `anomaly_id` is populated only at times labeled synthetic anomaly; `synthetic_event_id` marks the event interval, including negative controls. `duration` is the scheduled event interval in readings; it can differ from the number of positive labels.

`severity_sd` is **not applicable** to `dropout` or `stuck_sensor`, so these experiments use `NaN` severity instead of repeating equivalent experiments. A `spike` always lasts exactly one reading. Small Gaussian jitter is called a negative control by experimental definition, not because a model is guaranteed to ignore it.

## 5. Build experiment parameter combinations

Pilot settings create two deterministic replicates of each of ten fault types for each of three sensors, using one severity and usually five readings of duration. Full mode expands all applicable severities, durations, and replicates. Beware storage: exporting a separate **full copy of the selected benchmark source** for each full-mode experiment can require many GB.

In [ ]:
def experiment_specs(mode):
    durations = (PILOT_DURATION_SAMPLES,) if mode == "pilot" else FULL_DURATIONS_SAMPLES
    replicates = PILOT_REPLICATES if mode == "pilot" else FULL_REPLICATES
    for sensor in SENSOR_COLS:
        for fault_type in FAULT_TYPES:
            if fault_type == "gaussian_noise":
                severities = (0.05,) if mode == "pilot" else BENIGN_NOISE_SEVERITIES_SD
            elif fault_type in ("dropout", "stuck_sensor"):
                severities = (None,)
            else:
                severities = (PILOT_SEVERITY_SD,) if mode == "pilot" else FULL_SEVERITIES_SD

            valid_durations = (1,) if fault_type == "spike" else durations
            if fault_type == "short_burst":
                valid_durations = tuple(d for d in valid_durations if d >= 3)
            if fault_type == "intermittent_fault":
                valid_durations = tuple(d for d in valid_durations if d >= 2)
            for severity, duration, replicate in product(severities, valid_durations, range(1, replicates + 1)):
                yield {"sensor":sensor, "fault_type":fault_type,
                       "severity_sd":severity, "duration":int(duration), "replicate":replicate}

specifications = list(experiment_specs(MODE))
print(f"Planned {len(specifications):,} benchmark experiments ({MODE})")
display(pd.Series([r["fault_type"] for r in specifications], name="experiment_count").value_counts().rename_axis("fault_type").to_frame())
if MODE == "full":
    print("CAUTION: set SAVE_ALL_BENCHMARK_CSV=True only if disk capacity is sufficient.")

## 6. Core injection function

Each fault is generated from the **unchanged original dataset** using a stored random seed. The function never alters the original data. `start_index` is inclusive and `end_index` is inclusive; `duration` is measured in samples (one minute per reading).

In [ ]:
def inject_fault(clean_df, sensor, fault_type, start_index, duration, severity_sd, seed, event_id):
    """Inject exactly one synthetic event and return a new frame plus its event metadata."""
    if sensor not in SENSOR_COLS or fault_type not in FAULT_TYPES:
        raise ValueError("Unexpected sensor or fault type")
    start_index, duration = int(start_index), int(duration)
    end_exclusive = start_index + duration
    if duration < 1 or start_index < 1 or end_exclusive > len(clean_df):
        raise ValueError("Invalid fault interval")
    if not clean_df[FAILURE_HORIZON_COL].iloc[start_index:end_exclusive].eq(0).all():
        raise ValueError("Cannot inject during a PW_0.5h=1 pre-failure interval")
    if fault_type in ("dropout", "stuck_sensor"):
        severity_sd = None
    elif severity_sd is None or not np.isfinite(float(severity_sd)) or float(severity_sd) <= 0:
        raise ValueError("Positive severity required")

    rng = np.random.default_rng(int(seed))
    out = clean_df.copy(deep=True)
    values = out[sensor].to_numpy(dtype=float).copy()
    baseline = values[start_index:end_exclusive].copy()
    sd = REFERENCE_STATS[sensor]["std"]
    signed_direction = int(rng.choice([-1, 1]))
    amplitude = float(severity_sd) * sd if severity_sd is not None else None
    positive_phase = np.ones(duration, dtype=bool)

    if fault_type == "gaussian_noise":
        values[start_index:end_exclusive] += rng.normal(0, amplitude, duration)
        positive_phase[:] = False  # intended harmless measurement jitter
    elif fault_type == "spike":
        if duration != 1:
            raise ValueError("Spike must have duration 1")
        values[start_index] += signed_direction * amplitude
    elif fault_type == "short_burst":
        if duration < 3:
            raise ValueError("Short burst needs at least 3 readings")
        values[start_index:end_exclusive] += signed_direction * amplitude
    elif fault_type == "random_corruption":
        # Replace rather than add. Reference mean ± (1..2) × severity × SD.
        offsets = rng.uniform(1.0, 2.0, duration) * amplitude
        signs = rng.choice([-1.0, 1.0], size=duration)
        values[start_index:end_exclusive] = REFERENCE_STATS[sensor]["mean"] + signs * offsets
    elif fault_type == "dropout":
        values[start_index:end_exclusive] = np.nan
    elif fault_type == "bias":
        values[start_index:end_exclusive] += signed_direction * amplitude
    elif fault_type == "drift":
        # Ramp from just above zero to full severity to avoid a labeled-but-unmodified first step.
        ramp = np.linspace(0.0, 1.0, duration + 1)[1:]
        values[start_index:end_exclusive] += signed_direction * amplitude * ramp
    elif fault_type == "stuck_sensor":
        values[start_index:end_exclusive] = values[start_index - 1]
    elif fault_type == "increased_noise":
        values[start_index:end_exclusive] += rng.normal(0, amplitude, duration)
    elif fault_type == "intermittent_fault":
        positive_phase = (np.arange(duration) % 2 == 0)
        slc = values[start_index:end_exclusive].copy()
        slc[positive_phase] += signed_direction * amplitude
        values[start_index:end_exclusive] = slc

    # For some stuck-sensor placements, the held value equals normal readings in part
    # or all of the interval. Reject wholly invisible events (but not partial equality).
    changed_within = ~np.isclose(values[start_index:end_exclusive], baseline,
                                 atol=1e-12, rtol=0.0, equal_nan=True)
    if fault_type != "gaussian_noise" and not changed_within.any():
        raise ValueError("Invisible fault (no observable sensor change)")

    out[sensor] = values
    out["injection_interval"] = np.zeros(len(out), dtype='int8')
    out.loc[start_index:end_exclusive - 1, "injection_interval"] = 1
    out["synthetic_anomaly"] = np.zeros(len(out), dtype='int8')
    out.loc[start_index:end_exclusive - 1, "synthetic_anomaly"] = positive_phase.astype('int8')
    out["synthetic_event_id"] = pd.Series(pd.NA, index=out.index, dtype='string')
    out.loc[start_index:end_exclusive - 1, "synthetic_event_id"] = event_id
    out["anomaly_id"] = pd.Series(pd.NA, index=out.index, dtype='string')
    positive_indices = start_index + np.flatnonzero(positive_phase)
    out.loc[positive_indices, "anomaly_id"] = event_id

    metadata = {
        "experiment_id":event_id, "sensor":sensor, "fault_type":fault_type,
        "severity_sd":severity_sd, "duration":duration,
        "start_index":start_index, "end_index":end_exclusive - 1,
        "start_timestamp":str(clean_df.at[start_index, TIMESTAMP_COL]),
        "end_timestamp":str(clean_df.at[end_exclusive - 1, TIMESTAMP_COL]),
        "seed":int(seed), "direction":signed_direction if amplitude is not None else None,
        "n_labeled_readings":int(positive_phase.sum()),
        "n_changed_readings":int(changed_within.sum()),
        "is_negative_control":int(fault_type == "gaussian_noise"),
        "reference_source":reference_source,
        "benchmark_split":BENCHMARK_SPLIT,
        "baseline_sd":sd,
        "benchmark_csv":"",
    }
    return out, metadata

## 7. Choose valid starts and run pilot / full experiments

Starts are chosen deterministically with each experiment's seed. For `stuck_sensor`, the selected interval must exhibit at least one *observable* difference after holding the previous value; this matters because the selected source sensors have relatively few unique values. The first 59 readings of a candidate event's history must also carry `PW_0.5h = 0` for the maximum reading window of 60.

In [ ]:
def choose_start(sensor, fault_type, duration, seed):
    starts = eligible_cache.get(duration)
    if starts is None:
        starts = eligible_starts(duration)
    if fault_type == "stuck_sensor":
        arr = clean_benchmark[sensor].to_numpy()
        starts = np.asarray([
            s for s in starts if not np.all(np.isclose(arr[s:s+duration], arr[s-1], atol=1e-12, rtol=0))
        ], dtype=np.int64)
    if len(starts) == 0:
        raise ValueError(f"No valid position for {sensor}: {fault_type}, duration={duration}")
    # Independent RNG stream for start selection, separate from fault values.
    rng = np.random.default_rng(np.random.SeedSequence([RANDOM_SEED, int(seed), 991]))
    return int(rng.choice(starts))

source_snapshot = clean_benchmark.copy(deep=True)  # confirm injection never mutates the selected source

example_faults = {"spike", "drift", "dropout", "stuck_sensor"}
example_cases = {}
manifest_records = []
verification_records = []
if SAVE_ALL_BENCHMARK_CSV:
    (OUTPUT_DIR / "benchmarks").mkdir(parents=True, exist_ok=True)
if SAVE_EXAMPLE_CSV:
    (OUTPUT_DIR / "examples").mkdir(parents=True, exist_ok=True)

for experiment_number, spec in enumerate(specifications, start=1):
    event_id = f"E{experiment_number:06d}"
    experiment_seed = RANDOM_SEED + experiment_number  # stable per experiment
    start = choose_start(spec["sensor"], spec["fault_type"], spec["duration"], experiment_seed)
    benchmark, metadata = inject_fault(
        clean_benchmark, spec["sensor"], spec["fault_type"], start,
        spec["duration"], spec["severity_sd"], experiment_seed, event_id,
    )
    metadata["replicate"] = spec["replicate"]
    keep_example = (spec["sensor"] == SENSOR_COLS[0] and
                    spec["fault_type"] in example_faults and spec["replicate"] == 1)
    if SAVE_ALL_BENCHMARK_CSV:
        path = OUTPUT_DIR / "benchmarks" / f"{event_id}.csv"
        benchmark.to_csv(path, index=False)
        metadata["benchmark_csv"] = str(path)
    elif SAVE_EXAMPLE_CSV and keep_example:
        path = OUTPUT_DIR / "examples" / f"{event_id}.csv"
        benchmark.to_csv(path, index=False)
        metadata["benchmark_csv"] = str(path)

    if keep_example:
        example_cases[spec["fault_type"]] = (benchmark, metadata)
    manifest_records.append(metadata)

manifest = pd.DataFrame(manifest_records)
manifest_path = OUTPUT_DIR / "manifest.csv"
manifest.to_csv(manifest_path, index=False)
print(f"Created {len(manifest):,} experiments; manifest: {manifest_path.resolve()}")
print(f"CSV benchmark copies saved: {(manifest['benchmark_csv'].str.len() > 0).sum()} / {len(manifest)}")
display(manifest.head(8))

## 8. Verify the generated data

Automated checks ensure: (i) no original column outside the chosen sensor changes; (ii) there are no changes outside the injected interval; (iii) future-failure labels are preserved; (iv) positive labels and event IDs match each fault's intended active phases; (v) source data and timestamps remain intact; (vi) injections avoid intervals labeled `PW_0.5h = 1` (future failures within 30 minutes). These tests regenerate **every** experiment directly from its manifest seed and start index (even when its large CSV was not saved).

In [ ]:
def verify_benchmark(original, altered, record):
    start, end = int(record["start_index"]), int(record["end_index"])
    fault_type, sensor = record["fault_type"], record["sensor"]
    untouched = [c for c in original.columns if c != sensor]
    other_columns_unchanged = original[untouched].equals(altered[untouched])
    x, y = original[sensor].to_numpy(), altered[sensor].to_numpy()
    altered_positions = ~np.isclose(x, y, atol=1e-12, rtol=0.0, equal_nan=True)
    outside_unchanged = not altered_positions[:start].any() and not altered_positions[end+1:].any()
    interval = altered["injection_interval"].to_numpy()
    interval_correct = (interval.sum() == end-start+1 and interval[start:end+1].all())
    labels = altered["synthetic_anomaly"].to_numpy()
    expected_count = int(record["n_labeled_readings"])
    labels_correct = (labels.sum() == expected_count and not labels[:start].any() and not labels[end+1:].any())
    if fault_type == "gaussian_noise":
        labels_correct &= expected_count == 0
    elif fault_type == "intermittent_fault":
        expected_phases = (np.arange(end-start+1) % 2 == 0).astype(int)
        labels_correct &= np.array_equal(labels[start:end+1], expected_phases)
    else:
        labels_correct &= labels[start:end+1].all()
    ids_correct = (altered.loc[labels == 1, "anomaly_id"].eq(record["experiment_id"]).all()
                   and altered.loc[labels == 0, "anomaly_id"].isna().all())
    original_unchanged = source_snapshot.equals(original)  # source was not mutated
    outside_failure_horizon = altered.loc[start:end, FAILURE_HORIZON_COL].eq(0).all()
    count_correct = int(altered_positions.sum()) == int(record["n_changed_readings"])
    checks = dict(other_columns_unchanged=other_columns_unchanged, outside_unchanged=outside_unchanged,
                  interval_correct=interval_correct, labels_correct=labels_correct, ids_correct=ids_correct,
                  original_unchanged=original_unchanged, outside_failure_horizon=outside_failure_horizon,
                  changed_count_correct=count_correct)
    return checks

for rec in manifest.to_dict(orient="records"):
    updated, _ = inject_fault(
        clean_benchmark, rec["sensor"], rec["fault_type"], rec["start_index"],
        rec["duration"], None if pd.isna(rec["severity_sd"]) else rec["severity_sd"],
        rec["seed"], rec["experiment_id"],
    )
    checks = verify_benchmark(clean_benchmark, updated, rec)
    verification_records.append({"experiment_id":rec["experiment_id"], **checks,
                                 "passed": bool(all(checks.values()))})
verification = pd.DataFrame(verification_records)
verification_path = OUTPUT_DIR / "verification_summary.csv"
verification.to_csv(verification_path, index=False)
print(f"Verification: {verification['passed'].sum():,} / {len(verification):,} passed")
assert verification["passed"].all(), "Some benchmarks failed verification; inspect verification_summary.csv"
display(verification.head(6))

## 9. Visual inspection of injected examples

Compare the original sensor and its synthetic counterpart in the same timeline. The shaded span represents the **scheduled interval**, whereas the `synthetic_anomaly` label reflects the fault's positive phases. Dropouts appear as gaps in the synthetic trace.

In [ ]:
if PLOT_EXAMPLES:
    for kind in ("spike", "drift", "dropout", "stuck_sensor"):
        if kind not in example_cases:
            continue
        example_data, event = example_cases[kind]
        start, end = event["start_index"], event["end_index"]
        left, right = max(0, start - 12), min(len(clean_benchmark), end + 13)
        x_values = np.arange(left, right)
        fig, ax = plt.subplots(figsize=(10, 3.2))
        sensor = event["sensor"]
        ax.plot(x_values, clean_benchmark[sensor].iloc[left:right], label="Original", lw=1.8)
        ax.plot(x_values, example_data[sensor].iloc[left:right], label="Synthetic", lw=1.4, ls="--", marker=".")
        ax.axvspan(start-0.5, end+0.5, color="tab:orange", alpha=0.18, label="Injection interval")
        ax.set(title=f"{kind.replace('_', ' ').title()} | {sensor} | {event['experiment_id']}",
               xlabel="Reading index (1 minute per reading)", ylabel="Provided sensor scale")
        ax.legend(loc="best", fontsize=8)
        ax.grid(alpha=0.2)
        fig.tight_layout()
        plt.show()

## 10. Review the manifest (no ML metrics yet)

These are **experimental setup counts**, not precision, recall, or F1 results. All three splits were loaded in Section 2, but model training and evaluation are intentionally left for a separate notebook. The manifest includes `benchmark_split` so you can identify the source of each generated experiment.

In [ ]:
manifest_counts = (manifest.groupby(["fault_type", "sensor"], dropna=False)
                   .size().unstack(fill_value=0))
display(manifest_counts)
labels_by_fault = manifest.groupby("fault_type")["n_labeled_readings"].agg(["count", "sum", "min", "max"])
display(labels_by_fault.rename(columns={"sum":"total_positive_labels_across_copies"}))
ax = manifest["fault_type"].value_counts().sort_values().plot(kind="barh", figsize=(9, 4.3), legend=False)
ax.set(title=f"Number of synthetic benchmark experiments | {MODE} mode", xlabel="Experiment count", ylabel="Fault type")
plt.tight_layout()
plt.show()

## 11. Reading-window label alignment preview (no models)

For a causal window of length `RW`, the window ending at reading `t` contains `t-RW+1 ... t`. Future anomaly scores can be associated with the window's final timestamp. Two **synthetic sensor-anomaly** targets are provided: `synthetic_anomaly_at_end` and `synthetic_anomaly_any_in_window`. Do not mix these definitions when comparing reading windows.

`no_positive_failure_horizon_labels_in_window` means every reading in the window has `PW_0.5h = 0`; this is a conservative evaluation **filter** for separating injected sensor-fault detection from periods immediately preceding machine failure. It does **not** mean the machine is known to be healthy. Do **not** apply this exclusion when evaluating a model whose task is specifically predicting future machine failure, because it would remove its positive target periods.


In [ ]:
def align_reading_window_labels(benchmark_df, window):
    window = int(window)
    if not 1 <= window <= len(benchmark_df):
        raise ValueError("Bad reading-window size")
    label = benchmark_df["synthetic_anomaly"].to_numpy(dtype=np.int8)
    state = benchmark_df[FAILURE_HORIZON_COL].to_numpy(dtype=np.int8)
    end = np.arange(window-1, len(benchmark_df), dtype=int)
    # Cumulative counts avoid creating a large 2D sliding-window array.
    positives = np.r_[0, np.cumsum(label)]
    positive_horizon_prefix = np.r_[0, np.cumsum(state == 1)]
    labels_in_window = positives[end+1] - positives[end+1-window]
    positive_horizon_count = positive_horizon_prefix[end+1] - positive_horizon_prefix[end+1-window]
    return pd.DataFrame({
        "window": window,
        "timestamp_end": benchmark_df[TIMESTAMP_COL].iloc[end].to_numpy(),
        "reading_index_end": end,
        "synthetic_anomaly_at_end": label[end],
        "synthetic_anomaly_any_in_window": (labels_in_window > 0).astype('int8'),
        "no_positive_failure_horizon_labels_in_window": (positive_horizon_count == 0).astype('int8'),
    })

if example_cases:
    example_data, example_metadata = next(iter(example_cases.values()))
    aligned_parts = [align_reading_window_labels(example_data, window) for window in READING_WINDOWS]
    aligned = pd.concat(aligned_parts, ignore_index=True)
    # Keep a bounded excerpt near the injected event instead of exporting every aligned row.
    i = example_metadata["start_index"]
    alignment_excerpt = aligned[aligned["reading_index_end"].between(max(i-2, 0), i+6)].copy()
    alignment_path = OUTPUT_DIR / "reading_window_alignment_preview.csv"
    alignment_excerpt.to_csv(alignment_path, index=False)
    display(alignment_excerpt.head(20))
    print("Reading-window preview:", alignment_path.resolve())

## 12. Regeneration helper and next steps

CSV copies are optional because every dataset can be **recreated directly from its manifest record**. For example, to inspect any experiment, call `regenerate_from_manifest('E000010')` below. The function verifies the reconstructed copy against the source and its manifest record before returning it.

For the **next notebook**, fit preprocessing on the selected training baseline (recognizing that label 0 means no failure in 30 minutes, not confirmed healthy operation), train a normal-behaviour model (e.g., Isolation Forest) for each RW, and select thresholds with separate validation data. **For unbiased performance evaluation, regenerate benchmarks from the independent test split:** change `BENCHMARK_SPLIT` to `"test"` in Section 1 and rerun this generator. Never use the injected examples from the training split to evaluate a model fitted to that same split as if they were unseen data. Always distinguish future-failure labels from current synthetic sensor faults, do not feed `PW_0.5h` to the anomaly model as a feature, and compare reading windows using identical injected events.

In [ ]:
def regenerate_from_manifest(experiment_id):
    matching = manifest.loc[manifest["experiment_id"] == experiment_id]
    if len(matching) != 1:
        raise KeyError(f"Unknown experiment_id: {experiment_id}")
    row = matching.iloc[0]
    data, _ = inject_fault(
        clean_benchmark, row["sensor"], row["fault_type"], row["start_index"],
        row["duration"], None if pd.isna(row["severity_sd"]) else float(row["severity_sd"]),
        int(row["seed"]), experiment_id,
    )
    assert all(verify_benchmark(clean_benchmark, data, row).values())
    return data

recreated = regenerate_from_manifest(manifest.iloc[0]["experiment_id"])
print(f"Regenerated {manifest.iloc[0]['experiment_id']}: {len(recreated):,} rows, "
      f"{recreated['synthetic_anomaly'].sum()} positively labeled sensor-fault readings")

print("\nOutput files:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(OUTPUT_DIR), f"({p.stat().st_size / 1024:.1f} KiB)")

### Interpretation and limitations

1. The generated examples are **controlled synthetic sensor faults**, not evidence that any anomaly detector can identify them. No F1, PR-AUC, ROC-AUC, detection delay, or other performance scores are reported here.
2. `PW_0.5h = 1` means **machine failure is expected within the next 30 minutes**, not that the machine is necessarily currently faulty. `PW_0.5h = 0` means no such failure is indicated in that horizon; it is not a guarantee of healthy operation. The notebook conservatively avoids positive-label periods for injected events and for its baseline-SD calculations.
3. The future-failure label is retained unchanged for context and later prognosis experiments. **Do not use it as an input to a sensor-anomaly detector**, because it depends on the future. `synthetic_anomaly` is a separate ground-truth label for *injected* events, not a machine-failure outcome.
4. Baseline sensor SDs come from **training** rows labeled `PW_0.5h = 0`. Synthetic events are injected into **training** by default. Final held-out model evaluation must use an independent **test** split (`BENCHMARK_SPLIT = "test"`) rather than training-derived benchmarks.
5. The flag `no_positive_failure_horizon_labels_in_window` excludes reading windows containing positive pre-failure labels, **only when measuring isolated synthetic sensor-fault detection**. Do not use that exclusion for a machine-failure prediction task or you would discard positive examples.
6. Labels mark intended fault activity; due to discretized measurements, some `stuck_sensor` readings can equal the original even while the sensor is artificially frozen. Conversely, `gaussian_noise` modifies readings but is deliberately labeled 0 as a negative control.
7. The one-minute cadence, RW eligibility, and severity grid are configurable assumptions. Full-mode output can be very large; keep `SAVE_ALL_BENCHMARK_CSV=False` unless needed.
8. `dropout` introduces missing sensor values; future models need a consistent missing-value strategy fitted on training data, never held-out benchmarks.
